###  Ridge回帰とクロスバリデーションによる性能評価

簡単な例でRidge線形回帰とクロスバリデーションによる性能評価をおこないます。


In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler

%matplotlib inline

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 80)

# warning error を表示しない。
import warnings
warnings.filterwarnings('ignore')

In [ ]:
g_data_name = "x5_sin_noise"  # x5_sin, x123, x5_sin_noise
g_normalizationtype="standard" # standard, minmax
g_regtype = "lasso" # lasso, ridge
g_n_splits = 10 # CV分割数

In [ ]:
def get_data(data_name):
    """観測データの作成

    Args:
        data_name (str): 作成するデータの名前。

    Raises:
        ValueError: 規定外のdata_name。

    Returns:
        pd.DataFrame: 観測データ。
        pd.DataFrame: 新規データ
        List(str): 説明変数名のリスト
        str: 目的変数
    """        
    if data_name == "x5_sin":
        filename = "../data_calculated/x5_sin.csv"
        filename_new = "../data_calculated/x5_sin_new.csv"
        descriptor_names = ['x1', 'x2', 'x3', 'x4', 'x5', 'x6']
        # descriptor_names = ['x1', 'x2', 'x3', 'x4', 'x5', ]
        target_name = 'y'
    elif data_name == "x5_sin_noise":
        filename = "../data_calculated/x5_sin_noise.csv"
        filename_new = "../data_calculated/x5_sin_noise_new.csv"
        descriptor_names = ['x1', 'x2', 'x3', 'x4', 'x5', 'x6']
        # descriptor_names = ['x1', 'x2', 'x3', 'x4', 'x5', ]
        target_name = 'y'
    elif data_name == "x123":
        filename = "../data_calculated/x123.csv"
        filename_new = "../data_calculated/x123_new.csv"
        descriptor_names = ['x1', 'x2', 'x3']
        target_name = 'y'
        
    else:
        raise ValueError("unknown data_name={}".format(data_name))
    df_obs = pd.read_csv(filename)
    df_new = pd.read_csv(filename_new)
    return df_obs, df_new, descriptor_names, target_name

g_df_obs, g_df_new, g_descriptor_names, g_target_name = get_data(g_data_name)

# obs
g_Xraw = g_df_obs.loc[:, g_descriptor_names].values
g_y = g_df_obs.loc[:, g_target_name].values

# new 
g_Xraw_new = g_df_new.loc[:, g_descriptor_names].values
g_y_new = g_df_new.loc[:, g_target_name].values


def scale_X(Xraw, normalizationtype=None, scaler=None):
    """Xを規格化する。

    Args:
        Xraw (np.ndarray): 説明変数。
        normalizationtype (str, optional): 規格化の名前. Defaults to None.
        scaler (StandardScaler|MinMaxScaler, optional): 規格化クラスインスタンス. Defaults to None.

    Raises:
        ValueError: 規定外normalizationtype

    Returns:
        nd.ndarray: 規格化された説明変数

    """        
    if scaler is not None:
        print("use", scaler)
        X = scaler.transform(Xraw)
    else:
        print("normalizationtype", normalizationtype)
        if normalizationtype=="standard":
            from sklearn.preprocessing import StandardScaler
            scaler = StandardScaler()
            scaler.fit(Xraw)
            X = scaler.transform(Xraw)    
        elif normalizationtype=="mimax":
            from sklearn.preprocessing import MinMaxScaler
            scaler = MinMaxScaler()
            scaler.fit(Xraw)
            X = scaler.transform(Xraw)    
        elif normalizationtype is None:
            # 規格化を行わない。
            X = Xraw
            scaler = None
        else:
            raise ValueError("unkown normalizationtype={}".format(normalizationtype))
    return X, scaler

g_X, g_scaler = scale_X(g_Xraw, g_normalizationtype)
g_X_new, _ = scale_X(g_Xraw_new, scaler=g_scaler)

In [ ]:
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.model_selection import KFold


def choose_linear_model(regtype :str, alpha:float=1e-3):
    """線形モデルの選択を行う

    Args:
        regtype (str): 線形モデル名
        alpha (float, optional): Lasso, Ridgeのhyperparameter. Defaults to 1e-2.

    Raises:
        ValueError: 規定外線形モデル名。

    Returns:
        LinearRegression|Lasso|Ridge: 線型回帰モデルinstance
    """
    if regtype=="linear":
        reg = LinearRegression()
    elif regtype=="lasso":
        reg = Lasso(alpha=alpha)
    elif regtype=="ridge":
        reg = Ridge(alpha=alpha)
    else:
        raise ValueError("unkown regtype={}".format(regtype))
    return reg

def linear_regression_CV_score(X, y, alpha, regtype: str, n_splits=10):
    """Ridge regression with the CV score

    Args:
        X (np.array): descriptor
        y (np.array): target variable
        alpha (float): hyperparameter of Ridge regression
        n_splits (int, optional): the number of splits in CV. Defaults to 10.

    Returns:
        dict: the mean value of test scores of CV, the stddev value of test scores of CV
        
    """
    reg = choose_linear_model(regtype, alpha=alpha)
    kf = KFold(n_splits=n_splits, shuffle=True)
    test_score_list = []
    train_score_list =[]
    for train, test in kf.split(X):
        Xtrain, ytrain = X[train], y[train]
        Xtest, ytest = X[test], y[test]
        reg.fit(Xtrain, ytrain)
        test_score = reg.score(Xtest, ytest)
        test_score_list.append(test_score)
        train_score = reg.score(Xtrain, ytrain)
        train_score_list.append(train_score)
    return {"mean(testR2)":np.mean(test_score_list), "std(testR2)": np.std(test_score_list),
            "mean(trainR2)":np.mean(train_score_list), "std(trainR2)": np.std(train_score_list)}


今回はRidge回帰のハイパーパラメタαを動かし、
各αでCVを行う点が異なります。

In [ ]:
def CV_scores(X,y, regtype, alpha_list = np.logspace(-5, 2, 10)):
    """X yから回帰性能値を出す。

    Args:
        X (np.ndarray): 説明変数
        y (np.ndarray): 目的変数
        regtype (str): 回帰モデルの種類
        alpha_list (float, optional): 線形回帰モデルのハイパーパラメタ. Defaults to np.logspace(-5, 2, 10).

    Returns:
        pd.DataFrame: 回帰スコアのmean, std, alpha
    """
    test_mean_score_list = []
    test_std_score_list = []
    train_mean_score_list = []
    train_std_score_list = []    
    for alpha in alpha_list:
        result = linear_regression_CV_score(X, y, alpha, regtype=regtype)
        mean_score, std_score = result["mean(testR2)"], result["std(testR2)"]
        test_mean_score_list.append(mean_score)
        test_std_score_list.append(std_score)
        mean_score, std_score = result["mean(trainR2)"], result["std(trainR2)"]
        train_mean_score_list.append(mean_score)
        train_std_score_list.append(std_score)
        
    return pd.DataFrame({"alpha": alpha_list, 
                         "mean(testR2)":test_mean_score_list, "std(testR2)":test_std_score_list,
                        "mean(trainR2)": train_mean_score_list, "std(trainR2)": train_std_score_list})

g_df_score = CV_scores(g_X,g_y, g_regtype )
g_df_score

alpha_listに対する回帰スコアの表示を行います。

In [ ]:
def plot_apha_score(df : pd.DataFrame, show_detail: bool=False, save_fig: bool=False):
    """alphaの変化を図示する。

    Args:
        df (pd.DataFrame): 回帰スコアの平均値と標準偏差とalpha
        show_detail (bool, optiona): 0.99-1.0を拡大表示するか。Defaults to False.
    
    """
    alpha_list = df["alpha"]
    
    fig, ax = plt.subplots()
    
    mean_score_list = df["mean(testR2)"]
    std_score_list = df["std(testR2)"]    
    ax.errorbar(np.log10(alpha_list), mean_score_list,
                 yerr=std_score_list, fmt="o-", capsize=5, label="test")
    
    
    mean_score_list = df["mean(trainR2)"]
    std_score_list = df["std(trainR2)"]    
    ax.errorbar(np.log10(alpha_list), mean_score_list,
                 yerr=std_score_list, fmt="o-", capsize=5, label="train")
    
    ax.set_xlabel("log10(alpha)")
    ax.set_ylabel("$R^2$ (CV test)")
    ax.legend()
    if show_detail:
        # 0.99以上のみ表示する。
        plt.ylim((0.9, 1.01))
    if save_fig:
        import os
        os.makedirs("image_executed", exist_ok=True)
        fig.savefig("image_executed/alpha_vs_R2.png")
    fig.show()
    
plot_apha_score(g_df_score, save_fig=True)

R2がほぼ１なのでどれを選んでも良いのですが、np.argmax()で最大のmean(R2)のindexを取得して、alpha_optを得ます。

In [ ]:
g_dfs = g_df_score.sort_values(by="mean(testR2)", ascending=False).reset_index(drop=True)
display(g_dfs)
g_alpha_opt = g_dfs.loc[0,"alpha"]
print("alpha_opt",g_alpha_opt)

以上でCVによる
過学習しない範囲で最も予測性能の高い
最適なハイパーパラメタの値が分かりました。

過学習しないハイパーパラメタ値が分かったのでこのハイパーパラメタを用いて一つの回帰モデルを作りなおします。
（scikit-learnのRidgeCVにならい全観測データを用いて一つの回帰モデルを作り直します。

In [ ]:
g_regtype

In [ ]:
g_reg = choose_linear_model(g_regtype, alpha=g_alpha_opt)
g_reg.fit(g_X,g_y)
g_yp = g_reg.predict(g_X)

作り直したモデルに対して予測を行います。

まず、y vs ypのplotを行うための関数定義をします。

In [ ]:
def plot_y_yp(y,yp, title: str=None, save_fig: bool=False):
    """y vs ypを図示する。

    Args:
        y (np.ndarray): 目的変数値
        yp (np.ndarray): 目的変数予測値
        title (str, optional): 図のtitle. Defaults to None.
    """
    fig,ax = plt.subplots(figsize=(5,5))

    # $y^{obs}$ vs $y^{predict}$
    ax.plot(y,yp,"o")

    # 斜め線を引く
    yall = np.hstack([y,yp])
    ylim = yall.min(), yall.max()
    ax.plot(ylim,ylim,"--")

    # labelを書く
    ax.set_xlabel("$y_{obs}$")
    ax.set_ylabel("$y_{pred}$")
    if title is not None:
        ax.title(title)
    if save_fig:
        fig.savefig("image_executed/y_obs_vs_y_pred.png")
    fig.show()


作り直したモデルに対してcross validationの予測を行います。そして

- $y^{obs}$ vs $y^{pred}$のplot
- 係数のplot

を行います。

In [ ]:
def Ridge_regression_CV_yp_coef(alpha, X, y, regtype: str, n_splits=10, random_state=1):
    """linear regression with cross validation

    Args:
        alpha (float): hyperparameter
        X (np.array): descriptor
        y (np.array): target variable
        n_splits (int, optional): the number of splits in CV. Defaults to 10.
        random_state (int, optional): random state in KFold(). Defaults to 1.

    Returns:
        dict: y_test, predicted y_test, a list of linear coefficients
    """
    reg = choose_linear_model(regtype, alpha=alpha)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    coef_list = []
    ytest_list = []
    ytestp_list = []
    for train, test in kf.split(X):
        Xtrain, ytrain = X[train], y[train]
        Xtest, ytest = X[test], y[test]
        reg.fit(Xtrain, ytrain)
        ytestp = reg.predict(Xtest)
        ytest_list.append(ytest)
        ytestp_list.append(ytestp)
        coef_list.append(list(reg.coef_.ravel()))
    return {"ytest":ytest_list, "ytestp":ytestp_list, "coef":coef_list}

In [ ]:
g_result = Ridge_regression_CV_yp_coef(g_alpha_opt, g_X, g_y, 
                                regtype=g_regtype, n_splits=g_n_splits, random_state=1)
# y^obs vs y^pred
plot_y_yp(g_result["ytest"], g_result["ytestp"], save_fig=True)


In [ ]:
# 係数
def show_coef(coef):
    """線形モデルの係数の図示。

    Args:
        coef ([float]): 線形モデル係数
    """
    fig, ax = plt.subplots()
    dfcoef = pd.DataFrame(coef)
    dfcoef.plot(ax=ax)
    ax.set_xlabel("CV set index")
    
show_coef(g_result["coef"])

cross validation中でほぼ同じ線形回帰モデルが得られていることがわかります。

### 新規データに対する予測

In [ ]:

g_yp_new = g_reg.predict(g_X_new)


#### 可視化

In [ ]:
def plot_y_yp(y,yp, title: str=None):
    """y vs ypを図示する。

    Args:
        y (np.ndarray): 目的変数観測値
        yp (np.ndarray): s目的変数予測値
        title (str, optional): 図のtitle. Defaults to None.
    """
    fig, ax = plt.subplots(figsize=(5,5))

    # $y^{obs}$ vs $y^{predict}$
    ax.plot(y,yp,"o")

    # 斜め線を引く
    yall = np.hstack([y,yp])
    ylim = yall.min(), yall.max()
    ax.plot(ylim,ylim,"--")

    # labelを書く
    ax.set_xlabel("$y_{obs}$")
    ax.set_ylabel("$y_{pred}$")
    if title is not None:
        ax.set_title(title)
    fig.show()

plot_y_yp(g_result["ytest"],g_result["ytestp"],)
plot_y_yp(g_y_new, g_yp_new)

In [ ]:
def plotXy(X, y, yp, X_new, y_new, yp_new):
    """X, y, predicted y, X_new, y_new, predicted y_newの図示。

    Args:
        X (np.ndarray): 観測データ説明変数値
        y (np.ndarray): 観測データ目的変数観測値
        yp (np.ndarray): 観測データ目的変数予測値
        X_new (np.ndarray): 新規データ説明変数値
        y_new (np.ndarray): 新規データ目的変数観測値
        yp_new (np.ndarray): 新規データ目的変数予測値
    """
    fig, ax = plt.subplots()
    ax.plot(X[:,0],y,label="obs")
    ax.plot(X_new[:,0], y_new, label="new")

    ax.plot(X[:,0],yp,label="pred,obs")
    ax.plot(X_new[:,0], yp_new, label="pred,new")

    ax.set_xlabel("x1")
    ax.set_ylabel("y")
    ax.legend()
    
plotXy(g_X,g_y,g_yp, g_X_new,g_y_new, g_yp_new)

### 問題１

1. 別dataでの実行を行う。


2. ridgeでの実行を行う。